# Install Relevant Libraries

In [1]:
!pip install numpy-stl solidpython

In [2]:
import numpy as np
import math

import imageio # Make sure imageio is installed. Colab usually has it.
import io
from ipywidgets import fixed, VBox, interactive # 'fixed' is needed for interact when passing a static argument. VBox for layout.
from ipywidgets import interact, IntSlider, Output
from IPython.display import display, clear_output

from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.spatial import ConvexHull
from stl import mesh

# Build 3D Mesh Trimesh from Scratch (Dummy pyramid)

In [3]:
vertices = np.array([
  [-3, -3, 0],
  [ 3, -3, 0],
  [ 3,  3, 0],
  [-3,  3, 0],
  [ 0,  0, 3]
])
faces = ConvexHull(vertices).simplices

pyramid = mesh.Mesh(np.zeros(faces.shape[0], dtype=mesh.Mesh.dtype))
for i, f in enumerate(faces):
  for j in range(3):
    pyramid.vectors[i][j] = vertices[f[j], :]

# pyramid.save('pyramid.stl')

## Interactive Pyramid Rotation (Z-axis Scrubber)

In [4]:
def rotate_and_plot_pyramid(angle_degrees):
  # Create a fresh copy of the original pyramid for each rotation
  current_pyramid = mesh.Mesh(pyramid.data.copy())

  # Rotate the pyramid around the Z-axis using numpy-stl's rotate method
  current_pyramid.rotate([0.0, 0.0, 1.0], math.radians(angle_degrees))

  # --- Data Visualization: Rotated Pyramid ---
  fig = plt.figure()
  ax = fig.add_subplot(projection='3d')
  ax.add_collection3d(Poly3DCollection(current_pyramid.vectors, alpha=0.5, edgecolor='k'))

  # Set fixed axis limits for consistent visualization
  ax.set_xlim([-4, 4])
  ax.set_ylim([-4, 4])
  ax.set_zlim([-1, 4])

  ax.set_title(f'Rotated Pyramid (Z-axis, {angle_degrees}°)')
  return fig

## GIF Generation for Pyramid Rotation

In [5]:
print("Generating GIF... This may take a moment.")

# Define GIF parameters
duration = 10 # seconds for the entire GIF
fps = 20    # frames per second
num_frames = duration * fps

# Generate a sequence of angles for a full 360-degree rotation
# The 'endpoint=False' ensures the start and end frames are not duplicated.
angles = np.linspace(0, 360, num_frames, endpoint=False)

frames = []
for angle in angles:
  # Generate each frame by calling the core plotting function
  fig = rotate_and_plot_pyramid(angle)

  # Save the figure to an in-memory buffer (BytesIO) to avoid disk I/O for each frame
  buf = io.BytesIO()
  fig.savefig(buf, format='png')
  buf.seek(0)

  # Read the image from the buffer and append to the frames list
  frames.append(imageio.v2.imread(buf))

  # Close the figure to free up memory after it's been saved
  plt.close(fig)

# Save the collected frames as a GIF file
imageio.mimsave('pyramid_rotation_z_axis.gif', frames, fps=fps)
print("GIF 'pyramid_rotation_z_axis.gif' created!")

Generating GIF... This may take a moment.
GIF 'pyramid_rotation_z_axis.gif' created!


In [6]:
# Create an output widget to contain the interactive plot
output_widget = Output()

# Create an interactive slider widget for the rotation angle
rotation_slider = IntSlider(min=0, max=360, step=5, value=0, description='Rotation Angle (°):')

# Define a function to be called by interact that manages the output widget
def update_plot(angle_degrees):
  with output_widget:
    clear_output(wait=True) # Clear previous plot within the output widget
    fig = rotate_and_plot_pyramid(angle_degrees)
    display(fig)
    plt.close(fig) # Close the figure to free up memory after it's been displayed

# Use ipywidgets.interactive to create the interactive widget object
# This allows us to access its children later for layout
interactive_widget = interactive(update_plot, angle_degrees=rotation_slider)

# Display the interactive slider (which is the first child of the interactive_widget)
# and the dedicated output widget below it
display(VBox([interactive_widget.children[0], output_widget]))